In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

# Add the project root to Python's import path so modules in src/ can be imported.
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
orders = pd.read_csv(r'..\data\raw\olist_orders_dataset.csv', index_col='order_id', 
                        parse_dates=[3, 4, 5, 6, 7], date_format="%Y-%m-%d %H:%M:%S")
customers = pd.read_csv(r'..\data\raw\olist_customers_dataset.csv', index_col='customer_id')
geolocations = pd.read_csv(r'..\data\raw\olist_geolocation_dataset.csv', index_col='geolocation_zip_code_prefix')
sellers = pd.read_csv(r'..\data\raw\olist_sellers_dataset.csv', index_col='seller_id')
reviews = pd.read_csv(r"../data/processed/reviews_translated.csv", index_col='order_id', 
                        parse_dates=['review_creation_date', 'review_answer_timestamp'], 
                        date_format="%Y-%m-%d %H:%M:%S")
items = pd.read_csv(r'..\data\raw\olist_order_items_dataset.csv', index_col='order_id', 
                    parse_dates=['shipping_limit_date'], date_format="%Y-%m-%d %H:%M:%S")

## 1. Order Lifecycle Validation

**Verify whether missing timestamps are consistent with `order_status`.**

Missing timestamps become more common in the later stages of the order lifecycle. We hypothesize that this pattern reflects the progression of order_status rather than data quality issues. Consequently, once an order fails to reach a stage, timestamps for subsequent stages are also expected to be missing.

Split orders into complete and incomplete lifecycle records.

In [3]:
nonull_orders = orders.dropna(how='any')
na_orders = orders.loc[list(set(orders.index) - set(nonull_orders.index))]

Examine the order status of complete records.

In [4]:
nonull_orders['order_status'].value_counts()

order_status
delivered    96455
canceled         6
Name: count, dtype: int64

Inspect canceled orders with complete lifecycle timestamps.

In [5]:
nonull_orders[nonull_orders['order_status'] == 'canceled']

,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
order_id,,,,,,,
1950d777989f6a877539f53795b4c3c3,1bccb206de9f0f25adc6871a1bcf77b2,canceled,2018-02-19 19:48:52,2018-02-19 20:56:05,2018-02-20 19:57:13,2018-03-21 22:03:51,2018-03-09
dabf2b0e35b423f94618bf965fcb7514,5cdec0bb8cbdf53ffc8fdc212cd247c6,canceled,2016-10-09 00:56:52,2016-10-09 13:36:58,2016-10-13 13:36:59,2016-10-16 14:36:59,2016-11-30
770d331c84e5b214bd9dc70a10b829d0,6c57e6119369185e575b36712766b0ef,canceled,2016-10-07 14:52:30,2016-10-07 15:07:10,2016-10-11 15:07:11,2016-10-14 15:07:11,2016-11-29
8beb59392e21af5eb9547ae1a9938d06,bf609b5741f71697f65ce3852c5d2623,canceled,2016-10-08 20:17:50,2016-10-09 14:34:30,2016-10-14 22:45:26,2016-10-19 18:47:43,2016-11-30
65d1e226dfaeb8cdc42f665422522d14,70fc57eeae292675927697fe03ad3ff5,canceled,2016-10-03 21:01:41,2016-10-04 10:18:57,2016-10-25 12:14:28,2016-11-08 10:58:34,2016-11-25
2c45c33d2f9cb8ff8b1c86cc28c11c30,de4caa97afa80c8eeac2ff4c8da5b72e,canceled,2016-10-09 15:39:56,2016-10-10 10:40:49,2016-10-14 10:40:50,2016-11-09 14:53:50,2016-12-08


Summarize missing timestamps by order status.

In [6]:
na_orders.groupby("order_status")[[
    'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date'
    ]].agg(lambda x: x.isna().sum())

,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
order_status,,,
approved,0,2,2
canceled,141,550,619
created,5,5,5
delivered,14,2,8
invoiced,0,314,314
processing,0,301,301
shipped,0,0,1107
unavailable,0,609,609


**Order lifecycle and status consistency**

* Orders with complete timestamp information are predominantly classified as `delivered`, although a small number of `canceled` orders also contain complete timestamps.

* For orders with missing timestamps, the aggregate missingness pattern is broadly consistent with lifecycle progression.

* The main exceptions are a small number of `delivered` orders with missing lifecycle timestamps, indicating that `order_status` and timestamp completion are not perfectly equivalent. This requires consideration when constructing delivery-related datasets.

* Different `order_status` values correspond to different levels of lifecycle completion. Therefore, whether to include or exclude non-delivered orders should depend on the objective of the subsequent analysis.

## 2. Location Data Cleaning

**Check and correct customer, seller, and geolocation city information against official Brazilian municipality references where possible.**

The previous inspection showed that `zip_code_prefix` does not uniquely identify a city or coordinate. The free-form `city` field also contains inconsistencies and ambiguities that cannot be resolved from the Olist data alone. This workflow validates city information against official references, handles selected inconsistencies, and aggregates multiple coordinates into representative ZIP–municipality locations.

### 1. Load and Normalize Geolocation Data

Load the Olist geolocation table and rename its columns to shorter, consistent names. Since `zip_code_prefix` is not unique in the geolocation data, it is moved from the index back to a regular column.

In [7]:
geolocations.columns = pd.Index(['lat', 'lng', 'city', 'state'])
geolocations.index.name = 'zip_code_prefix'
geolocations = geolocations.reset_index()

`normalize_city()` creates a standardized representation of each city name for matching. It decodes malformed URL/HTML characters, converts text to lowercase, removes accents, replaces selected punctuation with spaces, and collapses repeated whitespace.

In [8]:
from src.geo_clean import normalize_city

The original city value is retained while `city_norm` is used for geographic matching.

In [9]:
geolocations["city_norm"] = (
    geolocations["city"]
    .map(normalize_city)
)

### 2. Prepare Official IBGE Municipality and District References

The municipality and district references come from the IBGE Brazilian Territorial Division (Divisão Territorial Brasileira, DTB). DTB provides the official annual lists of Brazilian municipalities, districts, and subdistricts together with their geographic codes and administrative hierarchy.

Official source: [IBGE — Divisão Territorial Brasileira (DTB)](https://www.ibge.gov.br/geociencias/organizacao-do-territorio/divisao-regional/23701-divisao-territorial-brasileira.html)  
2018 files: [IBGE — DTB 2018 downloads](https://geoftp.ibge.gov.br/organizacao_do_territorio/estrutura_territorial/divisao_territorial/2018/)


In [10]:
municipalities_ibge = pd.read_excel(
    r"../data/raw/RELATORIO_DTB_BRASIL_MUNICIPIO.xls"
)

districts_ibge = pd.read_excel(
    r"../data/raw/RELATORIO_DTB_BRASIL_DISTRITO.xls"
)

Rename the original Portuguese IBGE fields to consistent English names.

Keep only the administrative identifiers and names required for municipality resolution. `municipality_full_code` is retained as the official municipality identifier.

In [11]:
municipalities_ibge = municipalities_ibge.rename(columns={
    "UF": "state_code",
    "Nome_UF": "state_name",
    "Região Geográfica Intermediária": "intermediate_region_code",
    "Nome Região Geográfica Intermediária": "intermediate_region_name",
    "Região Geográfica Imediata": "immediate_region_code",
    "Nome Região Geográfica Imediata": "immediate_region_name",
    "Mesorregião Geográfica": "mesoregion_code",
    "Nome_Mesorregião": "mesoregion_name",
    "Microrregião Geográfica": "microregion_code",
    "Nome_Microrregião": "microregion_name",
    "Município": "municipality_code",
    "Código Município Completo": "municipality_full_code",
    "Nome_Município": "municipality_name",
})

districts_ibge = districts_ibge.rename(columns={
    "UF": "state_code",
    "Nome_UF": "state_name",
    "Região Geográfica Intermediária": "intermediate_region_code",
    "Nome Região Geográfica Intermediária": "intermediate_region_name",
    "Região Geográfica Imediata": "immediate_region_code",
    "Nome Região Geográfica Imediata": "immediate_region_name",
    "Mesorregião Geográfica": "mesoregion_code",
    "Nome_Mesorregião": "mesoregion_name",
    "Microrregião Geográfica": "microregion_code",
    "Nome_Microrregião": "microregion_name",
    "Município": "municipality_code",
    "Código Município Completo": "municipality_full_code",
    "Nome_Município": "municipality_name",
    "Distrito": "district_code",
    "Código de Distrito Completo": "district_full_code",
    "Nome_Distrito": "district_name",
})

municipalities_ibge = municipalities_ibge[
    [
        "state_code",
        "state_name",
        "municipality_full_code",
        "municipality_name",
    ]
].copy()

districts_ibge = districts_ibge[
    [
        "state_code",
        "state_name",
        "municipality_full_code",
        "municipality_name",
        "district_full_code",
        "district_name",
    ]
].copy()

municipalities_ibge.info()
districts_ibge.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5570 entries, 0 to 5569
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   state_code              5570 non-null   int64 
 1   state_name              5570 non-null   object
 2   municipality_full_code  5570 non-null   int64 
 3   municipality_name       5570 non-null   object
dtypes: int64(2), object(2)
memory usage: 174.2+ KB


,state_code,state_name,municipality_full_code,municipality_name,district_full_code,district_name
0,11,Rondônia,1100015,Alta Floresta D'Oeste,110001505,Alta Floresta D'Oeste
1,11,Rondônia,1100015,Alta Floresta D'Oeste,110001515,Filadélfia d'Oeste
2,11,Rondônia,1100015,Alta Floresta D'Oeste,110001520,Izidolândia
3,11,Rondônia,1100015,Alta Floresta D'Oeste,110001525,Nova Gease d'Oeste
4,11,Rondônia,1100015,Alta Floresta D'Oeste,110001530,Rolim de Moura do Guaporé


### 3. Perform Initial Municipality Resolution

Apply the same city-name normalization procedure to the official municipality and district references, allowing Olist and IBGE names to be compared in a common form.

In [12]:
municipalities_ibge["municipality_norm"] = (
    municipalities_ibge["municipality_name"]
    .map(normalize_city)
)

districts_ibge["municipality_norm"] = (
    districts_ibge["municipality_name"]
    .map(normalize_city)
)

districts_ibge["district_norm"] = (
    districts_ibge["district_name"]
    .map(normalize_city)
)

Convert Olist two-letter state abbreviations to the full state names used in the IBGE references. 

In [13]:
state_map = {
    "AC": "Acre",
    "AL": "Alagoas",
    "AP": "Amapá",
    "AM": "Amazonas",
    "BA": "Bahia",
    "CE": "Ceará",
    "DF": "Distrito Federal",
    "ES": "Espírito Santo",
    "GO": "Goiás",
    "MA": "Maranhão",
    "MT": "Mato Grosso",
    "MS": "Mato Grosso do Sul",
    "MG": "Minas Gerais",
    "PA": "Pará",
    "PB": "Paraíba",
    "PR": "Paraná",
    "PE": "Pernambuco",
    "PI": "Piauí",
    "RJ": "Rio de Janeiro",
    "RN": "Rio Grande do Norte",
    "RS": "Rio Grande do Sul",
    "RO": "Rondônia",
    "RR": "Roraima",
    "SC": "Santa Catarina",
    "SP": "São Paulo",
    "SE": "Sergipe",
    "TO": "Tocantins",
}

geolocations["state_name"] = geolocations["state"].map(state_map)

Check for unmapped states before removing the original abbreviation column, then remove exact duplicate geolocation records.

In [14]:
geolocations["state_name"].isna().sum()

np.int64(0)

In [15]:
geolocations.drop("state", inplace=True, axis=1)

In [16]:
geo_places = geolocations.drop_duplicates()  

`resolve_municipalities()` resolves Olist locality labels within each state using three strategies: exact municipality matches, unambiguous district matches, and district–parent municipality matches for labels such as "district (municipality)".

It returns:

- resolved: records that can be assigned confidently to an official municipality;  
- unresolved: records for which no safe municipality assignment can be made.

Resolved records receive `municipality_full_code`, `municipality_name`, and `match_type`.

In [17]:
from src.geo_clean import resolve_municipalities


geo_resolved0, geo_unresolved0 = resolve_municipalities(geo_places, municipalities_ibge, districts_ibge)

Inspect unresolved geolocation data.

In [18]:
geo_unresolved0.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2118 entries, 0 to 2126
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   zip_code_prefix  2118 non-null   int64  
 1   lat              2118 non-null   float64
 2   lng              2118 non-null   float64
 3   city             2118 non-null   object 
 4   city_norm        2118 non-null   object 
 5   state_name       2118 non-null   object 
dtypes: float64(2), int64(1), object(3)
memory usage: 115.8+ KB


More than 2,000 geolocation records remain unresolved against the official references, indicating a data-quality issue in this part of the dataset. The resolved geolocation table is also not used as a validation source for customer and seller municipality assignments, because the resolution process may standardize or alter location information, so its resulting location keys are not guaranteed to match the original `(zip_code_prefix, state, city)` values in the customer and seller tables. In addition, customer and seller locations may contain similar inconsistencies relative to the official references.

Therefore, customer and seller locations are resolved independently against the same official IBGE references, while the cleaned geolocation data are retained separately as the source of representative coordinates for later analysis.



### 4. Resolve Customer and Seller Locations

Apply the same state conversion and city-name normalization used for the geolocation table to customer and seller locations.

In [19]:
customers["state_name"] = customers["customer_state"].map(state_map)
customers["city_norm"] = customers["customer_city"].map(normalize_city)

sellers["city_norm"] = sellers["seller_city"].map(normalize_city)
sellers["state_name"] = sellers["seller_state"].map(state_map)

Check for unmapped states before removing the original state-abbreviation columns.

In [20]:
print("Unmatched states in customers:", customers["state_name"].isna().sum())
print("Unmatched states in sellers:", sellers["state_name"].isna().sum())

Unmatched states in customers: 0
Unmatched states in sellers: 0


In [21]:
customers.drop("customer_state", inplace=True, axis=1)
sellers.drop("seller_state", inplace=True, axis=1)

Standardize the customer and seller location schemas so that the same municipality-resolution functions can be applied consistently. Customer and seller IDs are temporarily restored as ordinary columns during resolution.

In [22]:
customers.rename(columns={
    "customer_zip_code_prefix": "zip_code_prefix",
    "customer_city": "city",
}, inplace=True)
customers.reset_index(inplace=True)

sellers.rename(columns={
    "seller_zip_code_prefix": "zip_code_prefix",
    "seller_city": "city",
}, inplace=True)
sellers.reset_index(inplace=True)

Resolve customer and seller locations independently against the same official IBGE municipality and district references used for the geolocation data. Records that remain unresolved are retained for the subsequent cleanup stages.

In [23]:
customers_resolved0, customers_unresolved0 = resolve_municipalities(customers, municipalities_ibge, districts_ibge)
sellers_resolved0, sellers_unresolved0 = resolve_municipalities(sellers, municipalities_ibge, districts_ibge)

In [24]:
print(
    "Customers unresolved:",
    len(customers_unresolved0),
    "/",
    len(customers),
)

print(
    "Sellers unresolved:",
    len(sellers_unresolved0),
    "/",
    len(sellers),
)

Customers unresolved: 244 / 99441
Sellers unresolved: 99 / 3095


Inspect a sample of unresolved records to identify recurring problems.

In [25]:
display(
    customers_unresolved0[
        [
            "zip_code_prefix",
            "city",
            "state_name",
            "city_norm",
        ]
    ].head(20)
)

display(
    sellers_unresolved0[
        [
            "zip_code_prefix",
            "city",
            "state_name",
            "city_norm",
        ]
    ].head(20)
)

,zip_code_prefix,city,state_name,city_norm
0,23970,parati,Rio de Janeiro,parati
1,6835,embu,São Paulo,embu
2,75570,bom jesus,Goiás,bom jesus
3,76974,espigao do oeste,Rondônia,espigao do oeste
4,85139,vitoria,Paraná,vitoria
5,48967,nucleo residencial pilar,Bahia,nucleo residencial pilar
6,37925,piumhii,Minas Gerais,piumhii
7,65137,maioba,Maranhão,maioba
8,71950,taguatinga,Distrito Federal,taguatinga
9,47310,santana do sobrado,Bahia,santana do sobrado


,zip_code_prefix,city,state_name,city_norm
0,95076,caxias do sul,São Paulo,caxias do sul
1,88501,lages - sc,Santa Catarina,lages sc
2,21210,rio de janeiro,Rio Grande do Norte,rio de janeiro
3,88330,balenario camboriu,Santa Catarina,balenario camboriu
4,85301,laranjeiras do sul,São Paulo,laranjeiras do sul
5,87360,goioere,São Paulo,goioere
6,15350,auriflama/sp,São Paulo,auriflama/sp
7,3407,sao paulo / sao paulo,São Paulo,sao paulo / sao paulo
8,86170,sertanopolis,São Paulo,sertanopolis
9,2051,sao pauo,São Paulo,sao pauo


### 5. Resolve Surface-Format Problems

`resolve_surface_city_names()` handles simple address-like city strings containing removable geographic qualifiers such as state names, state abbreviations, or country names.

For example:

`lages - sc`
→ `lages`

A cleaned candidate is accepted only when the removed qualifier is compatible with the recorded state and the resulting city matches an official municipality.

In [26]:
from src.geo_clean import resolve_surface_city_names

Only records that remain unresolved after the initial municipality/district matching are passed to this stage. Records that still cannot be resolved safely remain available for the historical-name matching step.

In [27]:
sellers_resolved1, sellers_unresolved1 = resolve_surface_city_names(sellers_unresolved0, municipalities_ibge)
customers_resolved1, customers_unresolved1 = resolve_surface_city_names(customers_unresolved0, municipalities_ibge)
geo_resolved1, geo_unresolved1 = resolve_surface_city_names(geo_unresolved0, municipalities_ibge)

Inspect how many additional records are recovered by the surface-format cleanup.

In [28]:
print(
    "Customers resolved by surface cleanup:",
    f"{len(customers_resolved1):,} / {len(customers_unresolved0):,}",
    f"({len(customers_resolved1) / len(customers_unresolved0):.2%})",
)

print(
    "Sellers resolved by surface cleanup:",
    f"{len(sellers_resolved1):,} / {len(sellers_unresolved0):,}",
    f"({len(sellers_resolved1) / len(sellers_unresolved0):.2%})",
)

print(
    "Geolocation records resolved by surface cleanup:",
    f"{len(geo_resolved1):,} / {len(geo_unresolved0):,}",
    f"({len(geo_resolved1) / len(geo_unresolved0):.2%})",
)

Customers resolved by surface cleanup: 0 / 244 (0.00%)
Sellers resolved by surface cleanup: 19 / 99 (19.19%)
Geolocation records resolved by surface cleanup: 2 / 2,118 (0.09%)


In [29]:
display(
    sellers_unresolved1[
        [
            "city",
            "city_norm",
            "state_name",
        ]
    ].head(20)
)

,city,city_norm,state_name
0,caxias do sul,caxias do sul,São Paulo
2,rio de janeiro,rio de janeiro,Rio Grande do Norte
3,balenario camboriu,balenario camboriu,Santa Catarina
4,laranjeiras do sul,laranjeiras do sul,São Paulo
5,goioere,goioere,São Paulo
8,sertanopolis,sertanopolis,São Paulo
9,sao pauo,sao pauo,São Paulo
10,marechal candido rondon,marechal candido rondon,São Paulo
11,rio de janeiro,rio de janeiro,São Paulo
12,bahia,bahia,Bahia


**Conclusion.** Surface-format cleanup has a limited overall effect, with most additional resolutions occurring among seller locations.

### 6. Resolve Historical Municipality Names

Some unresolved city names may correspond to municipalities whose official names or spellings have changed over time. To handle these cases, use the IBGE **Alterações Toponímicas Municipais** dataset, which records previous and current municipality names together with the associated state and change information.

Official source: [IBGE — Alterações Toponímicas Municipais](https://www.ibge.gov.br/geociencias/organizacao-do-territorio/estrutura-territorial/27336-alteracoes-toponimicas-municipais.html)

`prepare_toponym_reference()` converts the raw historical table into a standardized matching reference by normalizing the previous and current municipality names and preparing the associated state information.

In [30]:
from src.geo_clean import prepare_toponym_reference

with open(r"..\data\raw\Alteracoes_Toponimicas_Municipais_2025.xls", "rb") as f:
    top_df = pd.read_excel(f)

top_ref = prepare_toponym_reference(top_df, old_name_col = 'NOME_ANTERIOR', new_name_col='NOME_ATUAL', state_col='UF')

`resolve_historical_city_names()` then attempts to resolve records that remain unmatched after the initial and surface-format stages.

It:
1. searches `city_norm` in both the historical old-name and current-name fields;
2. checks whether each historical candidate belongs to the recorded state;
3. validates the resulting current municipality against the official municipality reference;
4. resolves the record only when exactly one distinct current municipality remains.

In [31]:
from src.geo_clean import resolve_historical_city_names

customers_resolved2, customers_unresolved2 = resolve_historical_city_names(customers_unresolved1, municipalities_ibge, top_ref)
sellers_resolved2, sellers_unresolved2 = resolve_historical_city_names(sellers_unresolved1, municipalities_ibge, top_ref)
geo_resolved2, geo_unresolved2 = resolve_historical_city_names(geo_unresolved1, municipalities_ibge, top_ref)

Inspect the additional records recovered through the historical municipality-name reference.

In [32]:
print(
    "Customers resolved historically:",
    f"{len(customers_resolved2):,} / {len(customers_unresolved1):,}",
    f"({len(customers_resolved2) / len(customers_unresolved1):.2%})",
)

print(
    "Sellers resolved historically:",
    f"{len(sellers_resolved2):,} / {len(sellers_unresolved1):,}",
    f"({len(sellers_resolved2) / len(sellers_unresolved1):.2%})",
)

print(
    "Geolocation records resolved historically:",
    f"{len(geo_resolved2):,} / {len(geo_unresolved1):,}",
    f"({len(geo_resolved2) / len(geo_unresolved1):.2%})",
)

Customers resolved historically: 59 / 244 (24.18%)
Sellers resolved historically: 2 / 80 (2.50%)
Geolocation records resolved historically: 983 / 2,116 (46.46%)


**Conclusion.** Historical-name matching substantially improves geolocation resolution and resolves some customer records, but contributes little to seller resolution.

### 7. Integrate Geographic Data for Analysis

After all conservative resolution methods have been applied, inspect the remaining unresolved records.

In [33]:
print(
    "Customers unresolved after all steps:",
    f"{len(customers_unresolved2):,} / {len(customers):,}",
    f"({len(customers_unresolved2) / len(customers):.2%})",
)

print(
    "Sellers unresolved after all steps:",
    f"{len(sellers_unresolved2):,} / {len(sellers):,}",
    f"({len(sellers_unresolved2) / len(sellers):.2%})",
)

print(
    "Geolocation records unresolved after all steps:",
    f"{len(geo_unresolved2):,} / {len(geo_places):,}",
    f"({len(geo_unresolved2) / len(geo_places):.2%})",
)

Customers unresolved after all steps: 185 / 99,441 (0.19%)
Sellers unresolved after all steps: 78 / 3,095 (2.52%)
Geolocation records unresolved after all steps: 1,133 / 738,332 (0.15%)


The remaining unresolved records account for only a small share of the customer and geolocation data, although the proportion is somewhat higher for sellers. Their issues are also heterogeneous, including spelling errors, inconsistent city–state combinations, abbreviations, and non-municipality values. Further automatic correction would therefore require increasingly case-specific or weaker assumptions. At this point, the remaining records are retained as unresolved rather than extending the cleaning process further.

In [34]:
display(
    sellers_unresolved2[
        [
            "city",
            "city_norm",
            "state_name",
        ]
    ].head(20)
)

,city,city_norm,state_name
0,caxias do sul,caxias do sul,São Paulo
2,rio de janeiro,rio de janeiro,Rio Grande do Norte
3,balenario camboriu,balenario camboriu,Santa Catarina
4,laranjeiras do sul,laranjeiras do sul,São Paulo
5,goioere,goioere,São Paulo
8,sertanopolis,sertanopolis,São Paulo
9,sao pauo,sao pauo,São Paulo
10,marechal candido rondon,marechal candido rondon,São Paulo
11,rio de janeiro,rio de janeiro,São Paulo
12,bahia,bahia,Bahia


Combine records resolved through initial matching, surface cleanup, and historical-name matching into final resolved datasets.

In [35]:
geo_resolved = pd.concat([geo_resolved0, geo_resolved1, geo_resolved2], ignore_index=True)
customers_resolved = pd.concat([customers_resolved0, customers_resolved1, customers_resolved2], ignore_index=True)
sellers_resolved = pd.concat([sellers_resolved0, sellers_resolved1, sellers_resolved2], ignore_index=True)

For the geolocation data, keep only unique resolved geographic observations before coordinate aggregation so duplicated source records do not receive additional weight.

In [36]:
geo_resolved_unique = (
    geo_resolved[
        [
            "zip_code_prefix",
            "state_name",
            "municipality_full_code",
            "municipality_name",
            "lat",
            "lng",
        ]
    ]
    .drop_duplicates()
)

Aggregate the deduplicated records by ZIP code prefix, state, and municipality. This preserves cases where the same ZIP prefix is associated with different municipalities while reducing multiple coordinate pairs within each combination to a representative location. Median latitude and longitude are used because they are less sensitive to extreme coordinate observations than the mean.

In [37]:
geo_lookup = (
    geo_resolved_unique
    .groupby(
        [
            "zip_code_prefix", 
            "state_name",
            "municipality_full_code",
            "municipality_name",
        ],
        as_index=False
    )
    .agg(
        lat=("lat", "median"),
        lng=("lng", "median"),
        n_unique_coordinates=("lat", "size"),
    )
)

geo_lookup = geo_lookup.rename(columns={
    "lat": "representative_lat",
    "lng": "representative_lng",
})

In [38]:
print(
    "Geo lookup rows:",
    len(geo_lookup)
)

Geo lookup rows: 19016


For customers and sellers, restore their IDs as indices and attach the successfully resolved municipality information back to the complete tables rather than replacing them with resolved-only subsets.

In [39]:
customers.set_index("customer_id", inplace=True)
customers_resolved.set_index("customer_id", inplace=True)

sellers.set_index("seller_id", inplace=True)
sellers_resolved.set_index("seller_id", inplace=True)

In [40]:
customers.loc[customers.index.isin(customers_resolved.index),'municipality_full_code'] = customers_resolved["municipality_full_code"]
customers.loc[customers.index.isin(customers_resolved.index),'municipality_name'] = customers_resolved["municipality_name"]

sellers.loc[sellers.index.isin(sellers_resolved.index),'municipality_full_code'] = sellers_resolved["municipality_full_code"]
sellers.loc[sellers.index.isin(sellers_resolved.index),'municipality_name'] = sellers_resolved["municipality_name"]

customers["municipality_full_code"] = (
    customers["municipality_full_code"]
    .astype("Int64")
)

sellers["municipality_full_code"] = (
    sellers["municipality_full_code"]
    .astype("Int64")
)

Add an explicit geographic-resolution status so unresolved customer and seller records remain available for non-geographic analyses, while their inclusion or exclusion from municipality-dependent analyses can be tracked.

In [41]:
customers["geo_status"] = np.where(
    customers["municipality_full_code"].notna(),
    "resolved",
    "unresolved",
)

sellers["geo_status"] = np.where(
    sellers["municipality_full_code"].notna(),
    "resolved",
    "unresolved",
)

### 8. Save Processed Geographic Data

Save the processed customer, seller, and geographic lookup tables in Parquet format for use in later analysis.

Customer and seller IDs are preserved as indices, while unresolved records remain in the full tables and can be identified through `geo_status`. This avoids repeating the geographic-cleaning workflow in subsequent notebooks.

In [42]:
customers.to_parquet(
    r"../data/processed/customers_geo_processed.parquet",
    index=True,
)

sellers.to_parquet(
    r"../data/processed/sellers_geo_processed.parquet",
    index=True,
)

geo_lookup.to_parquet(
    r"../data/processed/geo_lookup.parquet",
    index=False,
)

geo_unresolved2.to_parquet(
    r"../data/processed/geolocations_unresolved.parquet",
    index=False,
)

**Location cleaning and geographic coverage**

* Official municipality and district references, together with limited surface-format and historical-name handling, resolve nearly all customer and geolocation records, while a somewhat larger unresolved share remains among sellers.

* The remaining unresolved cases are retained rather than corrected through increasingly case-specific or uncertain assumptions.

* The processed customer and seller tables preserve all records with explicit geographic-resolution status, while resolved geolocation records are aggregated into representative ZIP–municipality coordinates for later analysis.


## 3. Review Linkage Validation and Entity Attribution

**Assess whether review records can be reliably linked to orders, products, sellers, and delivery experiences before using review-derived category scores in downstream analysis.**

The review dataset provides an overall review score and textual feedback linked to `order_id`. The review content has been used to construct separate Product, Seller, and Delivery scores. However, these category scores do not necessarily have a direct one-to-one relationship with the entities they describe: a review may be linked to multiple orders, an order may contain multiple reviews, and a single order may contain multiple products or sellers.

This section therefore examines the review–order and order–entity relationships to determine which category scores can be linked directly to a unique target, which can still be attributed after examining the remaining entity scope, and which must remain ambiguous.

### 1. Inspect Order–Review Cardinality

Examine the linkage between `order_id` and `review_id` in both directions.

**Question: How many review records are associated with each order?**

First, count the number of review records belonging to each `order_id`. This identifies orders represented by more than one row in the review dataset.

In [43]:
reviews_per_order = reviews.groupby(level="order_id").size()
reviews_per_order.sort_values(ascending=False)

order_id
03c939fd7fd3b38f8485a0f95798f1f6    3
8e17072ec97ce29f0e1f111e598b0c85    3
c88b1d1b157a9999ce368f218a407141    3
df56136b8031ecd28e200bb18e6ddb2e    3
7f13a20e25350f4a55fb2a7c9a2e8d88    2
                                   ..
ffe73fc1d73e03fdb50e63903ddfe882    1
ffe8851012fcdaf83de7b595fd5154b3    1
ffea20c7630343a6cd9e09858c1295cd    1
ffea406a848c8afe4dec22bf6290ba00    1
0008288aa423d2a3f00fcb17cd7d8719    1
Length: 98673, dtype: int64

In [44]:
reviews_per_order.value_counts().sort_index()

1    98126
2      543
3        4
Name: count, dtype: int64

**Question: How many distinct review IDs are associated with each order?**

Next, count the number of unique review_id values associated with each order. This distinguishes orders with multiple review records from orders that are actually linked to multiple review IDs.

In [45]:
review_ids_per_order = reviews.groupby(level="order_id")["review_id"].nunique()
review_ids_per_order.sort_values(ascending=False)

order_id
03c939fd7fd3b38f8485a0f95798f1f6    3
8e17072ec97ce29f0e1f111e598b0c85    3
c88b1d1b157a9999ce368f218a407141    3
df56136b8031ecd28e200bb18e6ddb2e    3
7f13a20e25350f4a55fb2a7c9a2e8d88    2
                                   ..
ffe73fc1d73e03fdb50e63903ddfe882    1
ffe8851012fcdaf83de7b595fd5154b3    1
ffea20c7630343a6cd9e09858c1295cd    1
ffea406a848c8afe4dec22bf6290ba00    1
0008288aa423d2a3f00fcb17cd7d8719    1
Name: review_id, Length: 98673, dtype: int64

In [46]:
review_ids_per_order.value_counts().sort_index()

review_id
1    98126
2      543
3        4
Name: count, dtype: int64

**Question: How many distinct orders are associated with each review ID?**

Finally, examine the reverse relationship by counting the number of unique order_id values associated with each review_id.

In [47]:
order_ids_per_review = (
    reviews.reset_index()
    .groupby("review_id")["order_id"]
    .nunique()
)

order_ids_per_review.sort_values(ascending=False)

review_id
4548534449b1f572e357211b90724f1b    3
32415bbf6e341d5d517080a796f79b5c    3
4d0e6dd087008d1f992d25ef6e1f619f    3
3415c9f764e478409e8e0660ae816dd2    3
69a1068c3128a14994e3e422e4539e04    3
                                   ..
ffe4b0eb9c831fb681f3606b8a2d4510    1
ffe4f02c09c187a1136a72a488b2b6c7    1
ffe55169612639d4355f30885db26072    1
ffe556cda2b136b51ede6d5ce5559a53    1
fffefe7a48d22f7b32046421062219d1    1
Name: order_id, Length: 98410, dtype: int64

In [48]:
order_ids_per_review.value_counts().sort_index()

order_id
1    97621
2      764
3       25
Name: count, dtype: int64

**Observation**

The inspection reveals non-unique linkage in both directions:
- some orders are associated with multiple distinct review IDs;
- some review IDs are associated with multiple distinct order IDs.

These two cases are investigated separately because they represent different linkage patterns and require different downstream treatment.

### 2. Investigate Multiple Reviews per Order

For orders associated with multiple review IDs, compare the review fields and item-level information.

**Question: Do reviews within the same order contain**
1. the same or different scores;
2. the same or different comments;
3. the same or different creation dates?

In [49]:
multi_review_id_orders = review_ids_per_order[
    review_ids_per_order > 1
].index

multi_review_ids_check = (
    reviews.loc[
        multi_review_id_orders,
        [
            "review_id",
            "review_score",
            "review_comment_title",
            "review_comment_en",
            "review_creation_date",
            "review_answer_timestamp",
        ],
    ]
    .sort_values(
        ["review_creation_date", "review_answer_timestamp"]
    )
    .sort_index()
)

display(multi_review_ids_check)

,review_id,review_score,review_comment_title,review_comment_en,review_creation_date,review_answer_timestamp
order_id,,,,,,
0035246a40f520710769010f752e7507,2a74b0559eb58fc1ff842ecc999594cb,5,NaN,I'm used to buying products by baratheon. Supe...,2017-08-25,2017-08-29 21:45:57
0035246a40f520710769010f752e7507,89a02c45c340aeeb1354a24e7d4b2c1e,5,NaN,NaN,2017-08-29,2017-08-30 01:59:12
013056cfe49763c6f66bda03396c5ee3,73413b847f63e02bc752b364f6d05ee9,4,NaN,NaN,2018-03-04,2018-03-05 17:02:00
013056cfe49763c6f66bda03396c5ee3,ab30810c29da5da8045216f0f62652a2,5,NaN,NaN,2018-02-22,2018-02-23 12:12:30
0176a6846bcb3b0d3aa3116a9a768597,d8e8c42271c8fb67b9dad95d98c8ff80,5,NaN,NaN,2017-12-30,2018-01-02 10:54:47
...,...,...,...,...,...,...
ff763b73e473d03c321bcd5a053316e8,5e78482ee783451be6026e5cf0c72de1,3,NaN,I don't know what's happened. The others have ...,2017-11-18,2017-11-18 09:02:48
ff850ba359507b996e8b2fbb26df8d03,39de8ad3a1a494fc68cc2d5382f052f4,5,NaN,Quick shipping...Product 100%,2017-08-16,2017-08-17 11:56:55
ff850ba359507b996e8b2fbb26df8d03,80f25f32c00540d49d57796fb6658535,5,NaN,"Fast shipping, product as described in the adv...",2017-08-22,2017-08-25 11:40:22


The number of unique values for each field is calculated within each order. A value of 1 indicates that the field is identical across the order's review records, while a value greater than 1 indicates that the records differ in that field.

In [50]:
multi_review_ids_summary = (
    multi_review_ids_check
    .groupby(level="order_id")
    .agg(
        n_review_records=("review_id", "size"),
        n_unique_review_ids=("review_id", "nunique"),
        n_unique_scores=(
            "review_score",
            lambda x: x.nunique(dropna=False),
        ),
        n_unique_comments=(
            "review_comment_en",
            lambda x: x.nunique(dropna=False),
        ),
        n_unique_titles=(
            "review_comment_title",
            lambda x: x.nunique(dropna=False),
        ),
        n_unique_creation_dates=(
            "review_creation_date",
            lambda x: x.nunique(dropna=False),
        ),
    )
)

multi_review_ids_summary.apply(lambda x: x.value_counts())

,n_review_records,n_unique_review_ids,n_unique_scores,n_unique_comments,n_unique_titles,n_unique_creation_dates
1,NaN,NaN,345.0,319,534.0,155
2,543.0,543.0,202.0,226,13.0,390
3,4.0,4.0,NaN,2,NaN,2


**Question: Do multiple reviews per order correspond to different items within the order?**

The numbers of items, distinct products, and sellers are examined for orders associated with multiple review IDs.

In [51]:
order_structure = (
    items.groupby("order_id")
    .agg(
        n_items=("order_item_id", "size"),
        n_unique_products=("product_id", "nunique"),
        n_unique_sellers=("seller_id", "nunique"),
    )
)

multi_review_ids_item = (
    multi_review_ids_summary
    .merge(
        order_structure,
        left_index=True,
        right_index=True,
        how="left",
        validate="1:1", 
    )
)

multi_review_ids_item[["n_items", "n_unique_products", "n_unique_sellers"]].apply(lambda x: x.value_counts()).sort_index()

,n_items,n_unique_products,n_unique_sellers
1.0,469,504.0,531.0
2.0,56,33.0,13.0
3.0,10,4.0,NaN
4.0,6,2.0,NaN
6.0,2,1.0,NaN
11.0,1,NaN,NaN


**Observation**

The review characteristics are heterogeneous across multiple-review orders. Multiple reviews per order therefore do not follow a single obvious pattern based on review scores, comments, titles, and creation dates.

The item structure also does not support a general one-review-per-item interpretation. In particular, many orders associated with multiple review IDs contain only one item, showing that the existence of multiple reviews cannot generally be explained by reviews referring to different items within the order.

Since the review data contain neither `order_item_id` nor `product_id`, individual reviews cannot in general be assigned to particular items merely because an order contains multiple reviews.

### 3. Investigate Review IDs Associated with Multiple Orders


For review IDs associated with multiple orders, compare the review score, comment, and creation date.

**Question: Do review records associated with the same `review_id` contain**:
- the same or different scores;
- the same or different comments;
- the same or different creation dates?

In [52]:
multi_order_review_ids = order_ids_per_review[
    order_ids_per_review > 1
].index

multi_order_ids_check = (
    reviews[
        reviews["review_id"].isin(multi_order_review_ids)
    ]
    .reset_index()[
        [
            "review_id",
            "order_id",
            "review_score",
            "review_comment_title",
            "review_comment_en",
            "review_creation_date",
            "review_answer_timestamp",   
        ]
    ]
    .set_index("review_id")
    .sort_index()
)

display(multi_order_ids_check)

,order_id,review_score,review_comment_title,review_comment_en,review_creation_date,review_answer_timestamp
review_id,,,,,,
00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,NaN,"The ""original HP"" 60XL cartridge is not recogn...",2018-03-07,2018-03-20 18:08:23
00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,NaN,"The ""original HP"" 60XL cartridge is not recogn...",2018-03-07,2018-03-20 18:08:23
0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,NaN,NaN,2017-09-21,2017-09-26 03:27:47
0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,NaN,NaN,2017-09-21,2017-09-26 03:27:47
0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,NaN,Product delivered in the package of the suppli...,2018-03-07,2018-03-08 03:00:53
...,...,...,...,...,...,...
fe5c833752953fed3209646f1f63b53c,4863e15fa53273cc7219c58f5ffda4fb,1,NaN,"I bought two products and both of them, even s...",2018-02-28,2018-02-28 13:57:52
ff2fc9e68f8aabfbe18d710b83aabd30,2da58e0a7dcfa4ce1e00fad9d03ca3b5,2,NaN,NaN,2018-03-17,2018-03-19 11:44:15
ff2fc9e68f8aabfbe18d710b83aabd30,1078d496cc6ab9a8e6f2be77abf5091b,2,NaN,NaN,2018-03-17,2018-03-19 11:44:15


The number of unique values for each field is calculated within each `review_id`. A value of 1 indicates that the field is identical across the records with the same `review_id`, while a value greater than 1 indicates that the records differ in that field.

In [53]:
multi_order_ids_summary = (
    multi_order_ids_check
    .groupby(level="review_id")
    .agg(
        n_review_records=("order_id", "size"),
        n_unique_order_ids=("order_id", "nunique"),
        n_unique_scores=(
            "review_score",
            lambda x: x.nunique(dropna=False),
        ),
        n_unique_comments=(
            "review_comment_en",
            lambda x: x.nunique(dropna=False),
        ),
        n_unique_titles=(
            "review_comment_title",
            lambda x: x.nunique(dropna=False),
        ),
        n_unique_creation_dates=(
            "review_creation_date",
            lambda x: x.nunique(dropna=False),
        ),
    )
)

multi_order_ids_summary.apply(lambda x: x.value_counts())

,n_review_records,n_unique_order_ids,n_unique_scores,n_unique_comments,n_unique_titles,n_unique_creation_dates
1,NaN,NaN,789.0,789.0,789.0,789.0
2,764.0,764.0,NaN,NaN,NaN,NaN
3,25.0,25.0,NaN,NaN,NaN,NaN


**Question: Do orders associated with the same `review_id` belong to the same customer and occur within a short time window?**


For each `review_id` associated with multiple orders, compare the `customer_unique_id` values of those orders and calculate the time span between their purchase timestamps. A five-minute threshold is also used to identify orders placed within a very short period of time.

In [54]:
customer_order_structure = (
    orders.reset_index()
    .merge(
        customers[["customer_unique_id"]],
        left_on="customer_id",
        right_index=True,
        how="left",
        validate="m:1",
    )
    .set_index("order_id")
)

multi_order_ids_customers = (
    multi_order_ids_check
    .merge(
        customer_order_structure,
        left_on="order_id",
        right_index=True,
        how="left",
    )[
        [
        "customer_unique_id",
        "order_purchase_timestamp"
        ]
    ]
    .groupby(level="review_id")
    .agg(
            n_unique_customers=("customer_unique_id", "nunique"),
            purchase_time_span=("order_purchase_timestamp", lambda x: (x.max() - x.min()))
    )
)

multi_order_ids_customers["purchase_within_5_minutes"] =  (multi_order_ids_customers["purchase_time_span"] <= pd.Timedelta(minutes=5))
multi_order_ids_customers[["n_unique_customers", "purchase_within_5_minutes"]].apply(lambda x: x.value_counts())


,n_unique_customers,purchase_within_5_minutes
False,NaN,48
1,789.0,741


**Question: Do orders associated with the same `review_id` contain the same products and sellers?**

For each `review_id` associated with multiple orders, compare every distinct pair of associated orders. The products and sellers within each order are represented as sets, allowing both exact equality and partial overlap to be examined.

In [55]:
multi_order_ids_items = (
    items[["product_id", "seller_id"]]
    .groupby(level="order_id")
    .agg(lambda x: frozenset(x))
    .merge(
        multi_order_ids_check[["order_id"]].drop_duplicates(),
        left_index=True,
        right_on="order_id",
        how="right",
    )
)

review_item_pairs = (
    multi_order_ids_items
    .merge(
        multi_order_ids_items,
        on="review_id",
        suffixes=("_1", "_2"),
    )
)

review_item_pairs = review_item_pairs[
    review_item_pairs["order_id_1"]
    < review_item_pairs["order_id_2"]
].copy()

In [56]:
review_item_pairs["has_item_data"] = (
    review_item_pairs["seller_id_1"].map(
        lambda x: isinstance(x, frozenset)
    )
    &
    review_item_pairs["seller_id_2"].map(
        lambda x: isinstance(x, frozenset)
    )
    & review_item_pairs["product_id_1"].map(
    lambda x: isinstance(x, frozenset)
    )
    & review_item_pairs["product_id_2"].map(
        lambda x: isinstance(x, frozenset)
    )
)

complete_pairs = review_item_pairs[ 
    review_item_pairs["has_item_data"]
].copy()


`same_sellers` indicates whether the two orders contain exactly the same sellers.  
`seller_overlap` indicates whether they share at least one seller.  
`same_products` indicates whether the two orders contain exactly the same products.  
`product_overlap` indicates whether they share at least one product.

In [57]:
complete_pairs["same_sellers"] = (
    complete_pairs["seller_id_1"]
    == complete_pairs["seller_id_2"]
)

complete_pairs["seller_overlap"] = [
    bool(a & b)
    for a, b in zip(
        complete_pairs["seller_id_1"],
        complete_pairs["seller_id_2"],
    )
]

complete_pairs["same_products"] = (
    complete_pairs["product_id_1"]
    == complete_pairs["product_id_2"]
)

complete_pairs["product_overlap"] = [
    bool(a & b)
    for a, b in zip(
        complete_pairs["product_id_1"],
        complete_pairs["product_id_2"],
    )
]

print(
    "Pairs with item data:",
    len(complete_pairs),
    "/",
    len(review_item_pairs),
)

complete_pairs[
    [
        "same_sellers",
        "seller_overlap",
        "same_products",
        "product_overlap",
    ]
].mean()

Pairs with item data: 665 / 729


same_sellers       0.330827
seller_overlap     0.345865
same_products      0.075188
product_overlap    0.075188
dtype: float64

**Observation**

Review IDs associated with multiple orders show a highly systematic pattern. Across all 789 such `review_id` groups, the review score, comment, title, and creation date are identical across the associated records. All associated orders also belong to the same customer, and 741 of the 789 groups (approximately 94%) contain orders purchased within five minutes of one another.

The associated orders also belong to the same customer, and approximately 94% of these `review_id` groups contain orders purchased within five minutes of one another.

Among order pairs with available item data, approximately 33% have identical seller sets and approximately 35% share at least one seller, indicating that most seller overlap consists of exact seller-set matches. In contrast, only about 8% of the order pairs share any product.

Thus, the same `review_id` is frequently linked to closely timed orders from the same customer, while the associated orders often contain different products and, in many cases, different sellers.

The available data do not reveal the mechanism behind this linkage. Possible explanations include shared review handling across related purchases, review–order linkage performed at a level not represented in the public dataset, or a systematic feature of the review or data-export process. The observed data are insufficient to distinguish reliably among these possibilities.


### 4. Assess Entity-Level Attribution

To prepare the review data for downstream analysis, determine whether each review can be linked reliably to a specific product, seller, or delivery experience. The resulting attribution table will be saved as an intermediate dataset and used later when combining review-derived category scores with the corresponding entities.

A target is **identifiable** when the review has a direct one-to-one relationship with that entity. When such a relationship is not available, the score may still be **attributable** if the remaining possible scope resolves to a single entity.

For example, a review associated with one order containing two different products does not identify a unique `product_id`. However, if both products are sold by the same seller, the Seller score can still be attributed to that `seller_id`. The Delivery score can likewise be attributed to the corresponding order–seller delivery experience. If multiple possible products or sellers remain, the corresponding score is treated as ambiguous.

<details> 
<summary><u>Attribution strategy and treatment of non-one-to-one relationships</u></summary>

The previous cardinality analysis revealed two different non-one-to-one relationships between reviews and orders. First, some `review_id` values are associated with multiple orders. These records are almost completely duplicated in their review content and are linked to closely timed orders from the same customer, but the data do not reveal which individual order the review refers to. These cases are therefore excluded from the current attribution analysis.

Second, some orders are associated with multiple distinct review IDs. Unlike the duplicated-review case, these records may contain different scores, comments, and timestamps, and may represent successive evaluations of the same order. They are therefore retained. When several reviews provide scores for the same category, the corresponding evaluations will be resolved later using the latest available rating within that category.

After resolving the review–order relationship, attribution still depends on the order-to-target cardinality. A single order may contain one or several products and sellers, so a review associated with one order does not necessarily identify a unique Product, Seller, or Delivery target. 

The following analysis therefore examines the product and seller structure within each retained order to determine which category scores can be attributed to a unique Product, Seller, or Delivery target and which remain ambiguous.
</details>

Count how many orders are associated with each review, and separate the structurally simple cases from review IDs spanning multiple orders.

In [58]:
review_order_links = (
    reviews
    .reset_index()[
        [
            "review_id",
            "order_id",
        ]
    ]
    .drop_duplicates()
)

review_order_structure = (
    review_order_links
    .groupby("review_id")
    .agg(
        n_orders=("order_id", "nunique"),
    )
)

single_order_review_ids = review_order_structure[
    review_order_structure["n_orders"] == 1
].index

multi_order_review_ids = review_order_structure[
    review_order_structure["n_orders"] > 1
].index

For each order, determine its items, distinct products, and distinct sellers.

In [59]:
order_entity_structure = (
    items
    .groupby(level="order_id")
    .agg(
        n_items=("order_item_id", "size"),
        n_unique_products=("product_id", "nunique"),
        n_unique_sellers=("seller_id", "nunique"),
        product_ids=(
            "product_id",
            lambda x: frozenset(x.dropna()),
        ),
        seller_ids=(
            "seller_id",
            lambda x: frozenset(x.dropna()),
        ),
    )
)

Keep only review IDs associated with exactly one order. Review IDs spanning multiple orders are set aside because their review-to-order scope is ambiguous and is not resolved in this stage.

In [60]:
single_order_reviews = (
    review_order_links[
        review_order_links["review_id"].isin(
            single_order_review_ids
        )
    ]
    .merge(
        order_entity_structure,
        left_on="order_id",
        right_index=True,
        how="left",
        validate="m:1",
    )
    .set_index("review_id")
)

single_order_reviews["item_data_available"] = (
    single_order_reviews["n_items"].notna()
)

A **Product** score can be attributed to a specific `product_id` only when the associated order contains exactly one distinct product.

If the order contains multiple distinct products, the review may still contain a meaningful Product evaluation, but the available data do not indicate which product it concerns.

In [61]:
single_order_reviews["product_attributable"] = (
    single_order_reviews["item_data_available"]
    & single_order_reviews["n_unique_products"].eq(1)
)

single_order_reviews["product_id"] = [
    next(iter(product_ids))
    if attributable
    else pd.NA
    for product_ids, attributable in zip(
        single_order_reviews["product_ids"],
        single_order_reviews["product_attributable"],
    )
]

A **Seller** score can be attributed to a specific `seller_id` when the associated order contains exactly one distinct seller. This remains possible even when the order contains multiple products, provided that all of them are associated with the same seller.

In [62]:
single_order_reviews["seller_attributable"] = (
    single_order_reviews["item_data_available"]
    &single_order_reviews["n_unique_sellers"].eq(1)
)

single_order_reviews["seller_id"] = [
    next(iter(seller_ids))
    if attributable
    else pd.NA
    for seller_ids, attributable in zip(
        single_order_reviews["seller_ids"],
        single_order_reviews["seller_attributable"],
    )
]

A **Delivery** score is attributed to an order–seller delivery experience. Since the analysis has already been restricted to reviews associated with exactly one order, a Delivery score is attributable when that order also contains exactly one distinct seller.

In [63]:
single_order_reviews["delivery_attributable"] = (
    single_order_reviews["item_data_available"]
    & single_order_reviews["n_unique_sellers"].eq(1)
)

single_order_reviews[
    [
        "order_id",
        "seller_id",
        "delivery_attributable",
    ]
]

,order_id,seller_id,delivery_attributable
review_id,,,
7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,6d803cb79cc31c41c4c789a75933b3c7,True
80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,8e6d7754bc7e0f22c96d255ebda59eba,True
228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,a1043bafd471dff536d0c462352beb48,True
e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,a1043bafd471dff536d0c462352beb48,True
f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,989becdce12ebc39863c2bceab6f3ca1,True
...,...,...,...
574ed12dd733e5fa530cfd4bbf39d7c9,2a8c23fee101d4d5662fa670396eb8da,5343d0649eca2a983820bfe93fc4d17e,True
f3897127253a9592a73be9bdfdf4ed7a,22ec9f0669f784db00fa86d035cf8602,46dc3b2cc0980fb8ec44634e21d2718e,True
b3de70c89b1510c4cd3d0649fd302472,55d4004744368f5571d1f590031933e4,17ca9b9e9b9ef8fdb529001b49ebb50f,True


Finally, retain the variables required for subsequent score attribution and summarize the proportions of reviews for which Product, Seller, and Delivery attribution is possible. 

In [64]:
review_entity_attribution = single_order_reviews[
    [
        "order_id",
        "n_items",
        "n_unique_products",
        "n_unique_sellers",
        "product_id",
        "seller_id",
        "product_attributable",
        "seller_attributable",
        "delivery_attributable",
    ]
].copy()


In [65]:
review_entity_attribution[
    [
        "product_attributable",
        "seller_attributable",
        "delivery_attributable",
    ]
].apply(lambda x: x.value_counts())

,product_attributable,seller_attributable,delivery_attributable
True,93786,95682,95682
False,3835,1939,1939


In [66]:
review_entity_attribution[
    [
        "product_attributable",
        "seller_attributable",
        "delivery_attributable",
    ]
].mean()

product_attributable     0.960715
seller_attributable      0.980137
delivery_attributable    0.980137
dtype: float64

**Observation**

After excluding review IDs associated with multiple orders, 97,621 review IDs remain for entity-attribution assessment. Of these, 696 (0.7%) cannot be evaluated because no corresponding item-level structure is available. Among the retained review IDs, 93,786 (96.1%) can be attributed to a specific Product target, while 95,682 (98.0%) can be attributed to Seller and Delivery targets.

The slightly lower Product attribution rate reflects orders containing multiple distinct products. In many of these cases, however, the products are associated with the same seller, allowing the Seller and Delivery evaluations to remain attributable even when the Product evaluation is ambiguous.

Overall, only a small share remain unattributable, allowing ambiguous cases to remain unassigned rather than forcing an arbitrary entity match. Orders associated with multiple review IDs are retained and will be handled separately when resolving the latest available evaluation within each category.


In [67]:
# Save `review_entity_attribution` for later use.

review_entity_attribution.to_parquet(
    r"../data/processed/review_entity_attribution.parquet",
    index=True,
)